# EWS NexusAI — End-to-End Pipeline

This notebook runs the full pipeline using the `src/` modules: load data, preprocess, train the Optuna-tuned LightGBM model, benchmark against baselines, and run the explainability / ablation / significance-testing suite.

**Note:** this replaces the original Colab notebook's `google.colab.files.upload()` step with a local CSV path — set `CSV_PATH` below before running.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import yaml

with open('../configs/default.yaml') as f:
    config = yaml.safe_load(f)

config

## 1. Load data

In [ ]:
from src.data.loader import load_bank_panel

CSV_PATH = config['data']['csv_path']  # update this path to your local CSV
df = load_bank_panel(CSV_PATH)
df.head()

## 2. Preprocess: clean columns + time-based train/test split

In [ ]:
from src.data.preprocessing import prepare_features_and_target, make_time_series_cv, get_feature_groups

split = prepare_features_and_target(
    df,
    target_col=config['data']['target_col'],
    test_fraction=config['data']['test_fraction'],
)
X_train, X_test, y_train, y_test = split.X_train, split.X_test, split.y_train, split.y_test
tscv = make_time_series_cv(n_splits=config['cv']['n_splits'])
X_train.shape, X_test.shape

## 3. Baseline LightGBM

In [ ]:
from src.models.lgb_model import fit_baseline_lgb, evaluate

baseline_lgb = fit_baseline_lgb(X_train, y_train, random_state=config['random_state'])
print('Baseline LightGBM:', evaluate(baseline_lgb, X_test, y_test))

## 4. Optuna-tuned LightGBM

In [ ]:
from src.models.lgb_model import tune_lgb_with_optuna, feature_importance_table

tuned_result = tune_lgb_with_optuna(
    X_train, y_train, X_test, y_test, cv=tscv,
    n_trials=config['optuna']['n_trials'], seed=config['optuna']['seed'],
)
tuned_lgb_model = tuned_result.model

print('Best params:', tuned_result.best_params)
print(f"CV mean MSE: {tuned_result.cv_mean_mse:.4f} [95% CI: {tuned_result.cv_ci[0]:.4f}, {tuned_result.cv_ci[1]:.4f}]")
print('Train:', evaluate(tuned_lgb_model, X_train, y_train))
print('Test:', evaluate(tuned_lgb_model, X_test, y_test))

In [ ]:
from src.utils.visualization import plot_feature_importance

imp_df = feature_importance_table(tuned_lgb_model, X_train.columns)
plot_feature_importance(imp_df, 'Feature Importances (Tuned LightGBM)')

## 5. SHAP explainability

In [ ]:
from src.utils.shap_explain import compute_shap_values, mean_abs_shap_table
import shap

shap_values = compute_shap_values(tuned_lgb_model, X_train, X_test)
mean_abs_shap_table(tuned_lgb_model and shap_values, X_test.columns).head(15)

In [ ]:
shap.summary_plot(shap_values, X_test, max_display=20)

## 6. Baseline model registry (Random Forest, Ridge, XGBoost) + benchmarking

In [ ]:
from src.models.baselines import build_model_registry, predict_all

models = build_model_registry(tuned_lgb_model, X_train, y_train)
predictions = predict_all(models, X_test)

for name, model in models.items():
    print(name, evaluate(model, X_test, y_test))

## 7. Feature-group ablation study (Accounting -> + Macro -> Full Features)

In [ ]:
from src.evaluation.ablation import run_ablation, pairwise_significance
from src.utils.visualization import plot_metric_with_ci

feature_groups = get_feature_groups()
ablation_result = run_ablation(
    tuned_lgb_model, X_train, X_test, y_train, y_test, feature_groups,
    n_bootstrap=config['evaluation']['ablation_n_bootstrap'],
)
ablation_result.summary_df

In [ ]:
plot_metric_with_ci(
    ablation_result.summary_df, x='Feature Set', y='R2',
    ci_lower_col='CI Lower', ci_upper_col='CI Upper',
    title='Ablation Study: Predictive Contribution of Feature Groups',
    ylabel='R2 Score (Test Set)', order=ablation_result.ordered_labels,
)

pairwise_significance(ablation_result, y_test)

## 8. ROC/AUC benchmark + Spearman correlation benchmark

In [ ]:
from src.evaluation.roc_and_correlation import (
    binarize_target, auroc_with_bootstrap_ci, pairwise_auroc_tests,
    spearman_benchmark, wilcoxon_compare,
)

y_test_binary = binarize_target(y_test, threshold=config['evaluation']['binarize_threshold'])
auc_df, auc_boot = auroc_with_bootstrap_ci(
    y_test_binary, predictions,
    n_bootstrap=config['evaluation']['n_bootstrap'],
)
auc_df

In [ ]:
pairwise_auroc_tests(auc_boot)

In [ ]:
spearman_df, spearman_boot = spearman_benchmark(
    y_test, predictions,
    n_bootstrap=config['evaluation']['n_bootstrap'],
)
spearman_df

## 9. Precision@K: does the model rank the riskiest banks correctly?

In [ ]:
from src.evaluation.precision_at_k import precision_at_k_benchmark, wilcoxon_vs_reference

precision_df, precision_boot = precision_at_k_benchmark(
    y_test, predictions,
    k_values=config['evaluation']['precision_at_k_values'],
    n_bootstrap=config['evaluation']['precision_at_k_n_bootstrap'],
)
precision_df

In [ ]:
wilcoxon_vs_reference(
    precision_boot, k_values=config['evaluation']['precision_at_k_values'],
    reference_model='Tuned LightGBM',
)